In [1]:
box::use(
  dada = dada2,
  bios = Biostrings,
  shor = ShortRead,
  stri = stringr,
  msa,
  ape,
  tidyverse[...], ggplot2[...], ggtree[...]
)

In [2]:
wdpath <- getwd()
demult.path <- paste0(wdpath, "/ku-invam/demultiplexed")
fns <- sort(list.files(demult.path, pattern = ".fastq.gz", full.names = TRUE))

In [4]:
#remove primers
# LSU_F <- "ACCCGCTGAACTTAAGC" #LROR
# LSU_R <- "GACGTAATGGCTTTAAACGA" #FLR2
SSU_LSU_F <- "TTGYTGCRGTTAAAAAGCTCG" #NS31Glo3
SSU_LSU_R <- dada$rc("AACACTCGCAYAYATGYTAGA") # LSUmBr

nops <- file.path(wdpath, "ku-invam/noprimers", basename(fns))
prim <- dada$removePrimers(
  fns, nops,
  primer.fwd = SSU_LSU_F,
  primer.rev = SSU_LSU_R,
  orient = TRUE
)
exists <- file.exists(nops)
paste("Sample", basename(nops[!exists]), "did not pass primer trimming")

Read in 1, output 0 (0%) filtered sequences.

Read in 1, output 0 (0%) filtered sequences.

Read in 1, output 0 (0%) filtered sequences.

Read in 1, output 0 (0%) filtered sequences.

Read in 1, output 0 (0%) filtered sequences.

Some input samples had no reads pass the primer detection.



[1] "Sample m64141e_241218_163908.hifi_reads.F2--R3.hifi_reads.fastq.gz did not pass primer trimming"
[2] "Sample m64141e_241218_163908.hifi_reads.F2--R7.hifi_reads.fastq.gz did not pass primer trimming"
[3] "Sample m64141e_241218_163908.hifi_reads.F6--R3.hifi_reads.fastq.gz did not pass primer trimming"
[4] "Sample m64141e_241218_163908.hifi_reads.F6--R7.hifi_reads.fastq.gz did not pass primer trimming"
[5] "Sample m64141e_241218_163908.hifi_reads.F7--R3.hifi_reads.fastq.gz did not pass primer trimming"

In [5]:
write.csv(prim, "ku-invam/primer_tracking.csv")

In [7]:
exists <- file.exists(nops)
nops <- nops[exists]
filts <- file.path(wdpath, "ku-invam/filtered", basename(nops))

track <- dada$filterAndTrim(
  nops, filts,
  minLen = 250, # Min length for sequence
  maxLen = 6000, # Max length for sequence
  rm.phix = FALSE, # No phix added (Illumina specific)
  qualityType = "FastqQuality", # Suggested for PacBio
  multithread = TRUE, # Allow multithread
  verbose = FALSE, # Print progress
  maxEE = 2 # Suggested default
)
exists <- file.exists(filts)
paste("Sample", basename(filts[!exists]), "did not pass filtering")
write.csv(track, "ku-invam/ku-invam_6000_filtering_tracking.csv")

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


Some input samples had no reads pass the filter.



[1] "Sample m64141e_241218_163908.hifi_reads.F1--R12.hifi_reads.fastq.gz did not pass filtering"
[2] "Sample m64141e_241218_163908.hifi_reads.F3--R13.hifi_reads.fastq.gz did not pass filtering"
[3] "Sample m64141e_241218_163908.hifi_reads.F4--R13.hifi_reads.fastq.gz did not pass filtering"
[4] "Sample m64141e_241218_163908.hifi_reads.F4--R2.hifi_reads.fastq.gz did not pass filtering" 
[5] "Sample m64141e_241218_163908.hifi_reads.F7--R12.hifi_reads.fastq.gz did not pass filtering"
[6] "Sample m64141e_241218_163908.hifi_reads.F7--R9.hifi_reads.fastq.gz did not pass filtering" 
[7] "Sample m64141e_241218_163908.hifi_reads.F8--R11.hifi_reads.fastq.gz did not pass filtering"

In [8]:
data.frame(
  "barcode" = sapply(
    filts[exists],
    function(x) {stri$str_split_i(basename(x), "[.]", 3)}
  ),
  "median" = sapply(
    filts[exists],
    function(x) {median(bios$width(bios$readDNAStringSet(x ,format='FASTQ')))}
  )
) |> write.csv("ku-invam/filtered_medians.csv")

In [9]:
exists <- file.exists(filts)
filts <- filts[exists]
drp <- dada$derepFastq(filts, verbose = TRUE)

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/ku-invam/filtered/m64141e_241218_163908.hifi_reads.F1--R1.hifi_reads.fastq.gz

Encountered 63 unique sequences from 184 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/ku-invam/filtered/m64141e_241218_163908.hifi_reads.F1--R11.hifi_reads.fastq.gz

Encountered 428 unique sequences from 1051 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/ku-invam/filtered/m64141e_241218_163908.hifi_reads.F1--R13.hifi_reads.fastq.gz

Encountered 1 unique sequences from 1 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/ku-invam/filtered/m64141e_241218_163908.hifi_reads.F1--R15.hifi_reads.fastq.gz

Encountered 267 unique sequences from 672 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Git

In [10]:
err <- dada$learnErrors(
  drp,
  errorEstimationFunction = dada$PacBioErrfun, # Set error estimation function for PacBio 
  multithread = TRUE, # Allow multithread
  BAND_SIZE = 32 # Suggested for Pacbio
)

91703933 total bases in 36554 reads from 72 samples will be used for learning the error rates.


In [11]:
dd <- dada$dada(
  drp,
  err = err,
  BAND_SIZE = 32,
  multithread = TRUE
)

Sample 1 - 184 reads in 63 unique sequences.
Sample 2 - 1051 reads in 428 unique sequences.
Sample 3 - 1 reads in 1 unique sequences.
Sample 4 - 672 reads in 267 unique sequences.
Sample 5 - 400 reads in 152 unique sequences.
Sample 6 - 631 reads in 179 unique sequences.
Sample 7 - 3 reads in 3 unique sequences.
Sample 8 - 107 reads in 32 unique sequences.
Sample 9 - 16 reads in 9 unique sequences.
Sample 10 - 1755 reads in 1160 unique sequences.
Sample 11 - 299 reads in 130 unique sequences.
Sample 12 - 1 reads in 1 unique sequences.
Sample 13 - 4 reads in 4 unique sequences.
Sample 14 - 2 reads in 2 unique sequences.
Sample 15 - 517 reads in 112 unique sequences.
Sample 16 - 255 reads in 35 unique sequences.
Sample 17 - 137 reads in 37 unique sequences.
Sample 18 - 1470 reads in 930 unique sequences.
Sample 19 - 1017 reads in 309 unique sequences.
Sample 20 - 74 reads in 28 unique sequences.
Sample 21 - 499 reads in 158 unique sequences.
Sample 22 - 573 reads in 197 unique sequences.

In [12]:
data.frame(
  "denoised" = sapply(
    dd,
    function(x) {sum(dada$getUniques(x))}
  )
) |> write.csv("ku-invam/denoised.csv")

In [13]:
st <- dada$makeSequenceTable(dd)
dim(st)

[1]  72 439

In [14]:
st.nobim <- dada$removeBimeraDenovo(
  st, method = "consensus",
  multithread = TRUE,
  verbose = TRUE
)

Identified 36 bimeras out of 439 input sequences.



In [15]:
data.frame(
  "nobimeras" = rowSums(st.nobim)
) |> write.csv("ku-invam/nobim.csv")

In [16]:
asv_table <- t(st.nobim)
colnames(asv_table) <- sapply(strsplit(rownames(st.nobim), "[.]"), `[`, 3)
rep_seqs <- bios$DNAStringSet(colnames(st.nobim))
rownames(asv_table) <- paste0("ASV_", seq(colnames(st.nobim)))
names(rep_seqs) <- rownames(asv_table)

In [18]:
sample_key <- read.csv("ku-invam/user_biosamples.csv")
old <- data.frame("Barcode" = colnames(asv_table))
new <- merge(old, sample_key, by = "Barcode")
old$Barcode == new$Barcode
new$SampleName <- paste0("Sample_", new$BioSample)
colnames(asv_table) <- new$SampleName


[1] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
[16] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
[31] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
[46] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
[61] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE

In [19]:
bios$writeXStringSet(rep_seqs, "ku-invam/ku-invam_rep_seqs.fasta")
write.table(
  asv_table,
  "ku-invam/ku-invam_asv_table.tsv",
  sep = "\t",
  row.names = TRUE,
  col.names = NA,
  quote = FALSE
)